In [1]:
import random
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torch.utils.tensorboard import SummaryWriter
from torchvision import datasets, transforms

In [2]:
IMAGES_DIR = "data/images"
BATCH_SIZE = 128
NUM_EPOCHS = 5
LR = 1e-3
WEIGHT_DECAY = 1e-4
TRAIN_VAL_SPLIT = 0.9
NUM_WORKERS = 4
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LOG_DIR = f"runs/imagenet_subset"

In [3]:
torch.manual_seed(SEED)
random.seed(SEED)

In [4]:
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(32),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize(36),
    transforms.CenterCrop(32),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

In [5]:
base_dataset = datasets.ImageFolder(root=IMAGES_DIR)  # used only to inspect classes/size
num_classes = len(base_dataset.classes)
total = len(base_dataset)
print(f"Found {num_classes} classes, {total} images total.")

indices = list(range(total))
random.shuffle(indices)
train_len = int(TRAIN_VAL_SPLIT * total)
train_indices = indices[:train_len]
val_indices = indices[train_len:]

train_dataset = Subset(datasets.ImageFolder(root=IMAGES_DIR, transform=train_transform), train_indices)
val_dataset = Subset(datasets.ImageFolder(root=IMAGES_DIR, transform=val_transform), val_indices)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)

Found 1000 classes, 3923 images total.


In [6]:
class CNNModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 96, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(96)

        self.conv2 = nn.Conv2d(96, 256, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(256)

        self.conv3 = nn.Conv2d(256, 384, 3, padding=1)
        self.conv4 = nn.Conv2d(384, 384, 3, padding=1)
        self.conv5 = nn.Conv2d(384, 256, 3, padding=1)

        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.5)

        self.fc1 = nn.Linear(256 * 4 * 4, 1024)
        self.fc2 = nn.Linear(1024, 512)
        self.fc3 = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.bn1(x)

        x = self.pool(F.relu(self.conv2(x)))
        x = self.bn2(x)

        x = F.relu(self.conv3(x))
        x = F.relu(self.conv4(x))
        x = F.relu(self.conv5(x))
        x = self.pool(x)

        x = torch.flatten(x, 1)
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.dropout(F.relu(self.fc2(x)))
        x = self.fc3(x)
        return x

In [7]:
model = CNNModel(num_classes=num_classes).to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

writer = SummaryWriter(log_dir=LOG_DIR)

best_val_acc = 0.0
global_step = 0
start_time = time.time()

In [8]:
for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    running_loss = 0.0
    running_corrects = 0
    total_train = 0

    for inputs, labels in train_loader:
        inputs = inputs.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        _, preds = torch.max(outputs, 1)
        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels).item()
        total_train += inputs.size(0)

        global_step += 1
        if global_step % 100 == 0:
            writer.add_scalar("train/batch_loss", loss.item(), global_step)
            lr = optimizer.param_groups[0]["lr"]
            writer.add_scalar("train/lr_batch", lr, global_step)

    epoch_loss = running_loss / total_train
    epoch_acc = running_corrects / total_train

    model.eval()
    val_loss = 0.0
    val_corrects = 0
    val_total = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = inputs.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            _, preds = torch.max(outputs, 1)
            val_loss += loss.item() * inputs.size(0)
            val_corrects += torch.sum(preds == labels).item()
            val_total += inputs.size(0)

    val_epoch_loss = val_loss / max(val_total, 1)
    val_epoch_acc = val_corrects / max(val_total, 1)

    if val_epoch_acc > best_val_acc:
        best_val_acc = val_epoch_acc

    writer.add_scalar("train/loss", epoch_loss, epoch)
    writer.add_scalar("train/accuracy", epoch_acc, epoch)
    writer.add_scalar("val/loss", val_epoch_loss, epoch)
    writer.add_scalar("val/accuracy", val_epoch_acc, epoch)
    writer.add_scalar("train/lr", optimizer.param_groups[0]["lr"], epoch)

    writer.add_histogram("params/conv1_weight", model.conv1.weight.detach().cpu().numpy(), epoch)
    writer.add_histogram("params/fc1_weight", model.fc1.weight.detach().cpu().numpy(), epoch)

    elapsed = time.time() - start_time
    print(f"Epoch {epoch}/{NUM_EPOCHS} "
          f"train_loss={epoch_loss:.4f} train_acc={epoch_acc:.4f} | "
          f"val_loss={val_epoch_loss:.4f} val_acc={val_epoch_acc:.4f} "
          f"elapsed={elapsed:.0f}s")

Epoch 1/5 train_loss=6.9125 train_acc=0.0020 | val_loss=6.9152 val_acc=0.0025 elapsed=34s
Epoch 2/5 train_loss=6.8860 train_acc=0.0017 | val_loss=6.9268 val_acc=0.0000 elapsed=62s
Epoch 3/5 train_loss=6.8535 train_acc=0.0031 | val_loss=6.9866 val_acc=0.0025 elapsed=90s
Epoch 4/5 train_loss=6.8137 train_acc=0.0014 | val_loss=6.9714 val_acc=0.0025 elapsed=118s
Epoch 5/5 train_loss=6.7782 train_acc=0.0048 | val_loss=6.9558 val_acc=0.0000 elapsed=147s


In [9]:
writer.add_text("meta", f"num_classes={num_classes}, total_images={total}", 0)
writer.close()

print(f"Training complete. Best val accuracy: {best_val_acc:.4f}")

Training complete. Best val accuracy: 0.0025
